In [22]:
import os
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ.pop("XLA_FLAGS", None)

import tensorflow as tf
tf.config.optimizer.set_jit(False)

In [23]:
import tensorflow as tf
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [24]:
import tensorflow as tf
from tensorflow.keras import layers

# ----------------------------
# PriorGuidedUpsample2D:
# UpSampling2D-based upsampling + Learnable prior fusion (NO pixelwise maximum)
# ----------------------------
@tf.keras.utils.register_keras_serializable(package="Custom")
class PriorGuidedUpsample2D(layers.Layer):
    """
    UpSampling2D upsampling everywhere (replaces PixelShuffleResize for features/skip and optionally priors)
    PLUS learnable prior fusion:

      GAP(pred) + GAP(prior) -> Dense -> Dense(1) -> project back (B,1,1,1) gate
      fused = gate*prior + (1-gate)*pred
      then DepthwiseConv(act) -> PointwiseConv to produce final map (B,H,W,1)

    Priors influence ONLY the structure maps (center/boundary), then those modulate features as before.
    """

    def __init__(
        self,
        scale=2,
        filters=64,
        method="bilinear",
        attention_kernel=3,
        use_structure_heads=True,
        n_centers=8,
        center_sharpness=9.0,
        boundary_suppress=0.4,
        learnable_temperature=True,
        learnable_prior_resize=True,
        # prior-fusion settings
        prior_fusion_hidden_ratio=4,  # hidden = filters//ratio
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.scale = int(scale)
        self.filters = int(filters)
        self.method = method
        self.attention_kernel = int(attention_kernel)
        self.use_structure_heads = bool(use_structure_heads)
        self.n_centers = int(n_centers)
        self.center_sharpness = float(center_sharpness)
        self.boundary_suppress = float(boundary_suppress)
        self.learnable_temperature = bool(learnable_temperature)

        self.learnable_prior_resize = bool(learnable_prior_resize)
        self.prior_fusion_hidden_ratio = int(prior_fusion_hidden_ratio)

        # ---------------- Projection & Attention ----------------
        self.proj_high = layers.Conv2D(filters, 1, padding="same", activation="selu", name="proj_high")
        self.proj_base = layers.Conv2D(filters, 1, padding="same", activation="linear", name="proj_base")
        self.skip_proj = layers.Conv2D(filters, 1, padding="same", use_bias=False, name="skip_proj")

        self.attention_head = tf.keras.Sequential(
            [
                layers.DepthwiseConv2D(attention_kernel, padding="same", activation="selu", use_bias=False),
                layers.Conv2D(filters, 1, padding="same", activation="selu"),
                layers.Conv2D(1, 1, padding="same", activation="sigmoid"),
            ],
            name="spatial_attention",
        )

        # ---------------- UpSampling2D-based upsamplers ----------------
        self.up_feat = layers.UpSampling2D(size=(self.scale, self.scale), interpolation=self.method, name="up_feat")
        self.up_feat_refine = tf.keras.Sequential(
            [
                layers.DepthwiseConv2D(3, padding="same", use_bias=False),
                layers.Conv2D(filters, 1, padding="same", activation="selu"),
            ],
            name="up_feat_refine",
        )

        if self.learnable_prior_resize:
            self.up_prior = layers.UpSampling2D(size=(self.scale, self.scale), interpolation=self.method, name="up_prior")
            self.up_prior_refine = tf.keras.Sequential(
                [
                    layers.DepthwiseConv2D(3, padding="same", use_bias=False),
                    layers.Conv2D(1, 1, padding="same", activation="selu"),
                ],
                name="up_prior_refine",
            )
        else:
            self.up_prior = None
            self.up_prior_refine = None

        # ---------------- Learned fusion scale ----------------
        self.fusion_scale = None
        self.alpha_weight = self.add_weight(
            shape=(1,), initializer=tf.keras.initializers.Constant(1.0), trainable=True, name="alpha_weight"
        )
        self.beta_weight = self.add_weight(
            shape=(1,), initializer=tf.keras.initializers.Constant(0.5), trainable=True, name="beta_weight"
        )

        # ---------------- Final projection (refinement step removed) ----------------
        self.final = layers.Conv2D(filters, 1, padding="same", activation="selu", name="final_head")

        # ---------------- Structural Heads ----------------
        self.assign_logits = layers.Conv2D(self.n_centers, 1, padding="same", use_bias=True)
        self.center_proj = layers.Conv2D(filters, 1, padding="same", activation=None)

        self.boundary_depthwise = layers.DepthwiseConv2D(3, padding="same", use_bias=False, name="laplace_dw")
        self.boundary_refine = tf.keras.Sequential(
            [
                layers.Conv2D(filters // 2, 3, padding="same", activation="selu"),
                layers.Conv2D(1, 1, padding="same", activation="sigmoid"),
            ],
            name="boundary_refine",
        )
        self.boundary_gate_net = tf.keras.Sequential(
            [
                layers.Conv2D(filters // 2, 3, padding="same", activation="selu"),
                layers.Conv2D(1, 1, padding="same", activation="sigmoid"),
            ],
            name="boundary_gate_net",
        )

        self.boundary_gate_scalar = None
        self.center_temp = None
        self.center_sigma = None

        # ---------------- Learnable prior fusion blocks (CENTER) ----------------
        hid = max(4, filters // self.prior_fusion_hidden_ratio)

        self.center_gap = layers.GlobalAveragePooling2D(name="center_prior_gap")
        self.center_fc1 = layers.Dense(hid, activation="selu", name="center_prior_fc1")
        self.center_gate = layers.Dense(1, activation="sigmoid", name="center_prior_gate")  # scalar gate

        self.center_dw = layers.DepthwiseConv2D(3, padding="same", activation="selu", name="center_prior_dw")
        self.center_pw = layers.Conv2D(1, 1, padding="same", activation="sigmoid", name="center_prior_pw")

        # ---------------- Learnable prior fusion blocks (BOUNDARY) ----------------
        self.boundary_gap = layers.GlobalAveragePooling2D(name="boundary_prior_gap")
        self.boundary_fc1 = layers.Dense(hid, activation="selu", name="boundary_prior_fc1")
        self.boundary_gate = layers.Dense(1, activation="sigmoid", name="boundary_prior_gate")  # scalar gate

        self.boundary_dw2 = layers.DepthwiseConv2D(3, padding="same", activation="selu", name="boundary_prior_dw")
        self.boundary_pw2 = layers.Conv2D(1, 1, padding="same", activation="sigmoid", name="boundary_prior_pw")

    # ---------------- Fusion scale initialization ----------------
    def _init_fusion_scale(self, shape, dtype):
        init = tf.keras.initializers.HeNormal(seed=54)
        w = init(shape=shape, dtype=dtype)
        return tf.clip_by_value(w, 0.0, 1.0)

    # ---------------- Build ----------------
    def build(self, input_shape):
        if not isinstance(input_shape, (list, tuple)) or len(input_shape) not in [2, 4]:
            raise ValueError(
                "PriorGuidedUpsample2D expects [low_res, skip_res] or "
                "[low_res, skip_res, prior_centers, prior_boundaries] inputs."
            )

        fusion_scale_init = self._init_fusion_scale((1, 1, 1, self.filters), tf.float32)
        self.fusion_scale = tf.Variable(
            initial_value=fusion_scale_init, trainable=True, name="fusion_scale", dtype=tf.float32
        )

        self.boundary_gate_scalar = self.add_weight(
            shape=(1,),
            initializer=tf.keras.initializers.Constant(1.0 - self.boundary_suppress),
            trainable=True,
            name="boundary_gate_scalar",
        )

        if self.learnable_temperature:
            self.center_temp = self.add_weight(
                shape=(1,),
                initializer=tf.keras.initializers.Constant(self.center_sharpness),
                trainable=True,
                name="center_temperature",
            )
        else:
            self.center_temp = tf.constant(self.center_sharpness, dtype=tf.float32)

        self.center_sigma = self.add_weight(
            shape=(self.n_centers,),
            initializer=tf.keras.initializers.Constant(1.0),
            trainable=True,
            name="center_sigma",
        )

        super().build(input_shape)

    # ---------------- Learnable upsample-to-target helper ----------------
    def _upsample_to(self, x, up_layer, refine, target_h, target_w):
        x = up_layer(x)
        if refine is not None:
            x = refine(x)
        # exact match by crop/pad (non-interpolating) for odd input sizes
        x = tf.image.resize_with_crop_or_pad(x, target_h, target_w)
        return x

    # ---------------- Learnable fusion (replaces pixelwise maximum) ----------------
    def _learnable_fuse_prior(self, pred_map, prior_map, gap, fc1, gate_fc, dw, pw):
        """
        pred_map/prior_map: (B,H,W,1)
        gate: scalar per-sample from GAP(pred)+GAP(prior)
        fused map refined by DWConv(act)+PWConv(sigmoid)
        """
        g_pred = gap(pred_map)   # (B,1) effectively after GAP; in TF it's (B,)
        g_prior = gap(prior_map)

        g = tf.concat([g_pred, g_prior], axis=-1)  # (B,2)
        h = fc1(g)
        gate = gate_fc(h)                          # (B,1)
        gate = tf.reshape(gate, [-1, 1, 1, 1])     # project back (broadcast)

        fused = gate * prior_map + (1.0 - gate) * pred_map
        fused = dw(fused)
        fused = pw(fused)
        return fused

    # ---------------- GMM-based Center & Boundary ----------------
    def _compute_soft_centers_gmm(self, feat, prior_centers=None, prior_boundaries=None):
        center_map, boundary_map = self._compute_centers_boundaries(feat)
        target_h, target_w = tf.shape(center_map)[1], tf.shape(center_map)[2]

        # --- learnable prior alignment + learnable fusion (NO tf.maximum) ---
        if prior_centers is not None:
            if self.up_prior is not None:
                prior_centers_up = self._upsample_to(
                    prior_centers, self.up_prior, self.up_prior_refine, target_h, target_w
                )
            else:
                prior_centers_up = tf.image.resize(prior_centers, (target_h, target_w), method="bilinear")
            prior_centers_up = tf.clip_by_value(prior_centers_up, 0.0, 1.0)

            center_map = self._learnable_fuse_prior(
                center_map, prior_centers_up,
                self.center_gap, self.center_fc1, self.center_gate,
                self.center_dw, self.center_pw
            )

        if prior_boundaries is not None:
            if self.up_prior is not None:
                prior_boundaries_up = self._upsample_to(
                    prior_boundaries, self.up_prior, self.up_prior_refine, target_h, target_w
                )
            else:
                prior_boundaries_up = tf.image.resize(prior_boundaries, (target_h, target_w), method="bilinear")
            prior_boundaries_up = tf.clip_by_value(prior_boundaries_up, 0.0, 1.0)

            boundary_map = self._learnable_fuse_prior(
                boundary_map, prior_boundaries_up,
                self.boundary_gap, self.boundary_fc1, self.boundary_gate,
                self.boundary_dw2, self.boundary_pw2
            )

        return center_map, boundary_map

    def _compute_centers_boundaries(self, feat):
        eps = 1e-9
        B = tf.shape(feat)[0]

        proj_feat = self.center_proj(feat)
        H, W, C = tf.shape(proj_feat)[1], tf.shape(proj_feat)[2], tf.shape(proj_feat)[3]
        feat_flat = tf.reshape(proj_feat, [B, H * W, C])

        logits = self.assign_logits(proj_feat)
        logits_flat = tf.reshape(logits, [B, H * W, self.n_centers])
        A_flat = tf.nn.softmax(logits_flat, axis=-1)

        A_flat_t = tf.transpose(A_flat, [0, 2, 1])
        denom = tf.reduce_sum(A_flat_t, axis=-1, keepdims=True) + eps
        prototypes = tf.matmul(A_flat_t, feat_flat) / denom

        sigma = tf.reshape(tf.maximum(self.center_sigma, 1e-6), [1, 1, self.n_centers, 1])

        feat_exp = tf.expand_dims(feat_flat, 2)
        proto_exp = tf.expand_dims(prototypes, 1)
        diff_sq = tf.reduce_sum((feat_exp - proto_exp) ** 2, axis=-1)
        gauss = tf.exp(-diff_sq / (2.0 * (sigma[..., 0] ** 2) + eps))
        resp = gauss / (tf.reduce_sum(gauss, axis=-1, keepdims=True) + eps)

        feat_norm = tf.nn.l2_normalize(feat_flat, axis=-1)
        proto_norm = tf.nn.l2_normalize(prototypes, axis=-1)
        sim = tf.matmul(feat_norm, proto_norm, transpose_b=True)

        weighted_sim = resp * sim
        per_pixel_strength = tf.reduce_max(weighted_sim, axis=-1, keepdims=True)

        sharpness = tf.maximum(self.center_temp, 1e-3)
        sharp = tf.pow(tf.nn.relu(per_pixel_strength) + eps, sharpness)
        max_img = tf.reduce_max(sharp, axis=[1, 2], keepdims=True) + eps
        center_map = sharp / max_img
        center_map = tf.reshape(center_map, [B, H, W, 1])

        resp_map = tf.reshape(tf.reduce_max(resp, axis=-1, keepdims=True), [B, H, W, 1])
        lap = self.boundary_depthwise(feat)
        lap_abs = tf.reduce_mean(tf.abs(lap), axis=-1, keepdims=True)
        refined_boundary = self.boundary_refine(lap_abs)
        spatial_gate = self.boundary_gate_net(feat)
        global_gate = tf.sigmoid(self.boundary_gate_scalar)
        boundary_map = refined_boundary * spatial_gate * global_gate * resp_map
        boundary_map = tf.clip_by_value(boundary_map, 0.0, 1.0)

        return tf.clip_by_value(center_map, 0.0, 1.0), boundary_map

    # ---------------- Forward ----------------
    def call(self, inputs, training=None):
        if len(inputs) == 2:
            low_res, skip_res = inputs
            prior_centers, prior_boundaries = None, None
        elif len(inputs) == 4:
            low_res, skip_res, prior_centers, prior_boundaries = inputs
        else:
            raise ValueError("Expected 2 or 4 inputs: [low_res, skip_res] optionally with priors.")

        high_proj = self.proj_high(low_res)
        base_proj = self.proj_base(low_res)
        alpha = self.attention_head(tf.concat([low_res, high_proj, base_proj], axis=-1))
        fused_low = alpha * high_proj + (1.0 - alpha) * base_proj

        target_h = tf.shape(low_res)[1] * self.scale
        target_w = tf.shape(low_res)[2] * self.scale

        # UpSampling2D-based upsampling (features + skip)
        upsampled = self._upsample_to(fused_low, self.up_feat, self.up_feat_refine, target_h, target_w)
        skip_proj = self.skip_proj(skip_res)
        skip_up = self._upsample_to(skip_proj, self.up_feat, self.up_feat_refine, target_h, target_w)

        fused = (upsampled + skip_up) * self.fusion_scale
        refined = fused

        if self.use_structure_heads:
            center_map, boundary_map = self._compute_soft_centers_gmm(refined, prior_centers, prior_boundaries)
            refined = refined * (self.alpha_weight + center_map) - boundary_map * self.beta_weight
            out = self.final(refined)
            return out, center_map, boundary_map

        out = self.final(refined)
        return out

    # ---------------- Config ----------------
    def get_config(self):
        cfg = super().get_config()
        cfg.update(
            {
                "scale": self.scale,
                "filters": self.filters,
                "method": self.method,
                "attention_kernel": self.attention_kernel,
                "use_structure_heads": self.use_structure_heads,
                "n_centers": self.n_centers,
                "center_sharpness": self.center_sharpness,
                "boundary_suppress": self.boundary_suppress,
                "learnable_temperature": self.learnable_temperature,
                "learnable_prior_resize": self.learnable_prior_resize,
                "prior_fusion_hidden_ratio": self.prior_fusion_hidden_ratio,
            }
        )
        return cfg

In [25]:
import tensorflow as tf
from tensorflow.keras import layers

@tf.keras.utils.register_keras_serializable(package="Custom")
class RadialWaveEvolutionaryConv(layers.Layer):

    def __init__(self,
                 num_patches=4,
                 num_feature_maps=32,
                 kernel_size=3,
                 **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.num_feature_maps = num_feature_maps
        self.kernel_size = kernel_size

    def build(self, input_shape):

        _, H, W, C = input_shape

        self.H = H
        self.W = W

        self.patch_h = H // self.num_patches
        self.patch_w = W // self.num_patches

        # Decision projection
        self.decision_filters = self.add_weight(
            shape=(C, self.num_feature_maps),
            initializer=tf.keras.initializers.GlorotNormal(),
            trainable=True,
            name="decision_space_filters"
        )

        # Convolution kernel
        self.gaussian_kernel = self.add_weight(
            shape=(self.kernel_size,
                   self.kernel_size,
                   C,
                   self.num_feature_maps),
            initializer=tf.keras.initializers.HeNormal(),
            trainable=True,
            name="gaussian_mutation_kernel"
        )

        # Learnable sigma (no user input)
        self.sigma = self.add_weight(
            shape=(),
            initializer=tf.keras.initializers.Constant(0.1),
            trainable=True,
            name="wave_sigma"
        )

        super().build(input_shape)

    # --------------------------------------------------
    # Pareto (tanh, batch-safe)
    # --------------------------------------------------
    def soft_pareto_score(self, objectives):
        # [B, N, M]

        f_i = tf.expand_dims(objectives, axis=2)  # [B, N, 1, M]
        f_j = tf.expand_dims(objectives, axis=1)  # [B, 1, N, M]

        domination = 0.5 * (1.0 + tf.nn.tanh(f_j - f_i))

        domination = tf.clip_by_value(domination, 1e-6, 1.0)

        domination_prod = tf.reduce_prod(domination, axis=-1)  # [B, N, N]

        score = tf.reduce_sum(domination_prod, axis=-1)  # [B, N]

        return score

    # --------------------------------------------------
    # Call
    # --------------------------------------------------
    def call(self, inputs):

        B = tf.shape(inputs)[0]
        C = tf.shape(inputs)[-1]

        # Patch extraction
        patches = tf.image.extract_patches(
            images=inputs,
            sizes=[1, self.patch_h, self.patch_w, 1],
            strides=[1, self.patch_h, self.patch_w, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        num_total_patches = self.num_patches * self.num_patches
        patch_area = self.patch_h * self.patch_w

        patches = tf.reshape(
            patches,
            [B, num_total_patches, patch_area, C]
        )

        patches_mean = tf.reduce_mean(patches, axis=2)  # [B, N, C]

        # Objectives
        objectives = tf.matmul(patches_mean, self.decision_filters)  # [B, N, M]

        # Pareto
        pareto_score = self.soft_pareto_score(objectives)

        weights = tf.nn.softmax(-pareto_score, axis=-1)
        weights = tf.expand_dims(weights, -1)

        weighted_objectives = objectives * weights

        # Radial wave
        center = tf.reduce_mean(weighted_objectives, axis=1, keepdims=True)

        radial_distance = tf.norm(weighted_objectives - center, axis=-1)

        sigma = tf.nn.softplus(self.sigma) + 1e-6

        # XLA-compatible approximation of Bessel J0 using cosine
        # cos(x / sqrt(2)) closely matches Bessel J0's behavior for small x
        wave = tf.math.cos(radial_distance / 1.41421356) 
        
        decay = tf.exp(- (radial_distance ** 2) / (2.0 * sigma ** 2))


        wave = wave * decay  # [B, N]

        # Expand to spatial
        wave = tf.image.resize(
            wave,
            size=(self.H, self.W),
            method="bilinear" # XLA fully supports this for both forward and backward pass
        )


        # Convolution
        conv_out = tf.nn.conv2d(
            inputs,
            self.gaussian_kernel,
            strides=1,
            padding="SAME"
        )

        return conv_out * wave

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "num_feature_maps": self.num_feature_maps,
            "kernel_size": self.kernel_size
        })
        return config

In [26]:
import tensorflow as tf
from tensorflow.keras import layers
import tensorflow.keras.backend as K

# --- 1. Combined Dice + IoU Loss ---
def dice_iou_loss(y_true, y_pred, smooth=1e-5):
    # Flatten
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    
    # Calculate Intersection and Union
    intersection = K.sum(y_true_f * y_pred_f)
    sum_ = K.sum(y_true_f) + K.sum(y_pred_f)
    
    # Dice
    dice = (2. * intersection + smooth) / (sum_ + smooth)
    dice_loss = 1.0 - dice
    
    # IoU (Jaccard)
    iou = (intersection + smooth) / (sum_ - intersection + smooth)
    iou_loss = 1.0 - iou
    
    # Total combined loss
    return dice_loss + iou_loss

# --- 2. Single Conv Block Helper ---
def single_conv_block(x, filters, kernel_size=3, padding="same", activation="relu"):
    """
    Standard UNet single convolution block: 1 Conv2D -> BatchNorm -> ReLU
    """
    x = layers.Conv2D(filters, kernel_size, padding=padding, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    return x

In [27]:
import tensorflow as tf
from tensorflow.keras import layers

# ---------------------------------------------------------
# Geodesic Kernel Downsampler
# ---------------------------------------------------------

@tf.keras.utils.register_keras_serializable(package="Custom")
class GeodesicDownsample2D(layers.Layer):
    """
    Riemannian geodesic pooling via iterative intrinsic mean computation
    with a learnable diagonal metric and configurable robust weighting
    kernels (Huber, Tukey, Cauchy, Geman-McClure).
    """

    def __init__(
        self,
        pool_size=2,
        geo_iters=3,
        geo_temp=1.0,
        robust_kernel="huber",
        tukey_c=4.685,
        huber_delta=1.0,
        cauchy_alpha=1.0,
        geman_lambda=1.0,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.pool_size = pool_size
        self.geo_iters = geo_iters
        self.geo_temp = geo_temp
        self.robust_kernel = robust_kernel.lower()

        # kernel params
        self.huber_delta = huber_delta
        self.tukey_c = tukey_c
        self.cauchy_alpha = cauchy_alpha
        self.geman_lambda = geman_lambda

    def build(self, input_shape):
        c = input_shape[-1]
        self.metric_raw = self.add_weight(
            shape=(c,),
            initializer=tf.keras.initializers.Constant(1.0),
            trainable=True,
            name="metric_diag_unconstrained"
        )
        super().build(input_shape)

    # ---------------- robust weighting functions ----------------
    def _robust_weights(self, dist_sq):
        eps = 1e-9

        if self.robust_kernel == "huber":
            delta = self.huber_delta
            mask = dist_sq <= delta**2
            w_quad = 1.0
            w_lin = delta / (tf.sqrt(dist_sq) + eps)
            return tf.where(mask, w_quad, w_lin)

        if self.robust_kernel == "tukey":
            c = self.tukey_c
            mask = dist_sq <= c**2
            z = dist_sq / (c**2 + eps)
            w = (1 - z)**2
            return tf.where(mask, w, tf.zeros_like(dist_sq))

        if self.robust_kernel == "cauchy":
            a = self.cauchy_alpha
            return 1.0 / (1.0 + dist_sq / (a**2) + eps)

        if self.robust_kernel == "geman":
            lam = self.geman_lambda
            return 1.0 / ((1.0 + dist_sq / (lam + eps))**2)

        return tf.nn.softmax(-dist_sq, axis=-1)

    # ---------------- main pooling routine ----------------
    def call(self, x):
        if isinstance(self.pool_size, int):
            ph = pw = self.pool_size
        else:
            ph, pw = self.pool_size

        b, h, w, c = tf.unstack(tf.shape(x))
        pad_h = tf.math.floormod(-h, ph)
        pad_w = tf.math.floormod(-w, pw)
        x_pad = tf.pad(x, [[0,0],[0,pad_h],[0,pad_w],[0,0]])

        Hp = tf.shape(x_pad)[1] // ph
        Wp = tf.shape(x_pad)[2] // pw

        ksizes = [1, ph, pw, 1]
        strides = [1, ph, pw, 1]
        patches = tf.image.extract_patches(
            images=x_pad,
            sizes=ksizes,
            strides=strides,
            rates=[1,1,1,1],
            padding="VALID"
        )

        P = ph * pw
        patches = tf.reshape(patches, [b, Hp, Wp, P, c])

        metric = tf.nn.softplus(self.metric_raw) + 1e-6
        mean = tf.reduce_mean(patches, axis=3)

        for _ in range(self.geo_iters):
            diff = patches - tf.expand_dims(mean, axis=3)
            dist_sq = tf.reduce_sum(diff * diff * tf.reshape(metric, [1,1,1,1,c]), axis=-1)
            w = self._robust_weights(dist_sq)
            w = tf.nn.softmax(w / self.geo_temp, axis=-1)
            w = tf.expand_dims(w, axis=-1)
            mean = tf.reduce_sum(patches * w, axis=3)

        return mean

    def get_config(self):
        base = super().get_config()
        base.update({
            "pool_size": self.pool_size,
            "geo_iters": self.geo_iters,
            "geo_temp": self.geo_temp,
            "robust_kernel": self.robust_kernel,
            "tukey_c": self.tukey_c,
            "huber_delta": self.huber_delta,
            "cauchy_alpha": self.cauchy_alpha,
            "geman_lambda": self.geman_lambda
        })
        return base



# ---------------------------------------------------------
# PriorGeneratingDownsample2D
#
# Changes from the original:
#   1. No more GatedSpatialAttention refinement step.
#   2. No more separate `refine_feat` input tensor. The skip
#      branch is now an average-pooled version of `in_feat`
#      itself, computed inside the layer with tf.nn.avg_pool2d
#      using the same pool_size as the geodesic downsampler.
#   3. call() now takes a single tensor `in_feat` instead of
#      a (in_feat, refine_feat) tuple, and build() expects a
#      single input_shape instead of a tuple of shapes.
#
# This layer produces the `center_map` / `boundary_map` priors
# (alongside the downsampled features) that PriorGuidedUpsample2D
# later consumes as `prior_centers` / `prior_boundaries`.
# ---------------------------------------------------------
@tf.keras.utils.register_keras_serializable(package="Custom")
class PriorGeneratingDownsample2D(layers.Layer):

    def __init__(self,
                 pool_size=2,
                 filters=64,
                 attention_kernel=3,
                 use_structure_heads=True,
                 n_centers=8,
                 center_sharpness=9.0,
                 boundary_suppress=0.4,
                 learnable_temperature=True,
                 **kwargs):

        super().__init__(**kwargs)

        self.pool_size = pool_size
        self.filters = filters
        self.attention_kernel = attention_kernel
        self.use_structure_heads = use_structure_heads
        self.n_centers = n_centers
        self.center_sharpness = center_sharpness
        self.boundary_suppress = boundary_suppress
        self.learnable_temperature = learnable_temperature

        # Replace pooling with geodesic kernel
        self.geo_pool = GeodesicDownsample2D(pool_size=pool_size)

        # Projections
        self.proj_high = layers.Conv2D(filters, 1, padding='same', activation='selu')
        self.proj_base = layers.Conv2D(filters, 1, padding='same')
        self.skip_proj = layers.Conv2D(filters, 1, padding='same')

        # Spatial attention
        self.attention_head = tf.keras.Sequential([
            layers.DepthwiseConv2D(attention_kernel, padding='same', activation='selu'),
            layers.Conv2D(filters, 1, padding='same', activation='selu'),
            layers.Conv2D(1, 1, padding='same', activation='sigmoid')
        ])

        # Fusion scale
        self.fusion_scale = None
        self.alpha_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True)
        self.beta_weight = self.add_weight(shape=(1,), initializer=tf.keras.initializers.Constant(0.5), trainable=True)

        # Final projection (refinement step removed)
        self.final = layers.Conv2D(filters, 3, padding='same', activation='selu')

        # Structure heads
        self.assign_logits = layers.Conv2D(self.n_centers, 1, padding='same')
        self.center_proj = layers.Conv2D(filters, 1, padding='same')

        self.boundary_depthwise = layers.DepthwiseConv2D(3, padding='same')
        self.boundary_refine = tf.keras.Sequential([
            layers.Conv2D(filters // 2, 3, padding='same', activation='selu'),
            layers.Conv2D(1, 1, padding='same', activation='sigmoid')
        ])
        self.boundary_gate_net = tf.keras.Sequential([
            layers.Conv2D(filters // 2, 3, padding='same', activation='selu'),
            layers.Conv2D(1, 1, padding='same', activation='sigmoid')
        ])

    def build(self, input_shape):
        # input_shape is now a single tensor shape (in_feat only)
        _, _, _, C = input_shape

        fusion_scale_init = tf.keras.initializers.HeNormal(seed=54)(shape=(1, 1, 1, self.filters))
        self.fusion_scale = tf.Variable(fusion_scale_init, trainable=True, name='fusion_scale')

        self.boundary_gate_scalar = self.add_weight(
            shape=(1,),
            initializer=tf.keras.initializers.Constant(1.0 - self.boundary_suppress),
            trainable=True
        )

        if self.learnable_temperature:
            self.center_temp = self.add_weight(
                shape=(1,),
                initializer=tf.keras.initializers.Constant(self.center_sharpness),
                trainable=True
            )
        else:
            self.center_temp = tf.constant(self.center_sharpness)

        self.center_sigma = self.add_weight(
            shape=(self.n_centers,),
            initializer='ones',
            trainable=True
        )

        super().build(input_shape)

    # ------------------------------------------------------
    # GMM structural mask logic (unchanged)
    # ------------------------------------------------------
    def _compute_soft_centers_gmm(self, feat):
        eps = 1e-9
        B = tf.shape(feat)[0]
        proj = self.center_proj(feat)
        H = tf.shape(proj)[1]
        W = tf.shape(proj)[2]
        C = tf.shape(proj)[3]

        flat = tf.reshape(proj, [B, H * W, C])
        logits = self.assign_logits(proj)
        logits_flat = tf.reshape(logits, [B, H * W, self.n_centers])
        A = tf.nn.softmax(logits_flat, axis=-1)
        A_t = tf.transpose(A, [0, 2, 1])
        denom = tf.reduce_sum(A_t, axis=-1, keepdims=True) + eps
        proto = tf.matmul(A_t, flat) / denom

        sigma = tf.reshape(tf.maximum(self.center_sigma, 1e-6), [1, 1, self.n_centers, 1])
        feat_exp = tf.expand_dims(flat, 2)
        proto_exp = tf.expand_dims(proto, 1)
        diff_sq = tf.reduce_sum((feat_exp - proto_exp) ** 2, axis=-1)
        gauss = tf.exp(-diff_sq / (2.0 * (sigma[..., 0] ** 2) + eps))
        resp = gauss / (tf.reduce_sum(gauss, axis=-1, keepdims=True) + eps)

        feat_n = tf.nn.l2_normalize(flat, axis=-1)
        proto_n = tf.nn.l2_normalize(proto, axis=-1)
        sim = tf.matmul(feat_n, proto_n, transpose_b=True)

        weighted_sim = resp * sim
        strength = tf.reduce_max(weighted_sim, axis=-1, keepdims=True)
        sharpness = tf.maximum(self.center_temp, 1e-3)
        sharp = tf.pow(tf.nn.relu(strength) + eps, sharpness)
        max_img = tf.reduce_max(sharp, axis=[1, 2], keepdims=True) + eps
        center_map = sharp / max_img
        center_map = tf.reshape(center_map, [B, H, W, 1])

        resp_map = tf.reshape(tf.reduce_max(resp, axis=-1, keepdims=True), [B, H, W, 1])
        lap = self.boundary_depthwise(feat)
        lap_abs = tf.reduce_mean(tf.abs(lap), axis=-1, keepdims=True)
        refined_boundary = self.boundary_refine(lap_abs)
        spatial_gate = self.boundary_gate_net(feat)

        boundary_map = refined_boundary * spatial_gate * tf.sigmoid(self.boundary_gate_scalar) * resp_map
        boundary_map = tf.clip_by_value(boundary_map, 0.0, 1.0)

        return tf.clip_by_value(center_map, 0.0, 1.0), boundary_map

    # ------------------------------------------------------
    # Forward path with geodesic pooling
    #
    # `in_feat` is now the ONLY input. The skip branch is
    # derived internally via average pooling of `in_feat`
    # (replacing the old separate `refine_feat` argument),
    # and the GatedSpatialAttention refinement step has been
    # removed entirely — the merged features flow straight
    # into the structure heads / final projection.
    # ------------------------------------------------------
    def call(self, in_feat):
        if isinstance(self.pool_size, int):
            ph = pw = self.pool_size
        else:
            ph, pw = self.pool_size

        pooled = self.geo_pool(in_feat)

        # Average-pooled skip branch, derived directly from in_feat
        pooled_ref = tf.nn.avg_pool2d(
            in_feat,
            ksize=[1, ph, pw, 1],
            strides=[1, ph, pw, 1],
            padding='SAME'
        )

        if tf.shape(pooled_ref)[1] != tf.shape(pooled)[1] or tf.shape(pooled_ref)[2] != tf.shape(pooled)[2]:
            pooled_ref = tf.image.resize(pooled_ref, tf.shape(pooled)[1:3], method='bilinear')

        high = self.proj_high(pooled)
        base = self.proj_base(pooled)
        alpha = self.attention_head(tf.concat([pooled, high, base], axis=-1))
        fused = alpha * high + (1 - alpha) * base

        skip = self.skip_proj(pooled_ref)
        fused = (fused + skip) * self.fusion_scale

        if self.use_structure_heads:
            center_map, boundary_map = self._compute_soft_centers_gmm(fused)
            fused = fused * (self.alpha_weight + center_map) - boundary_map * self.beta_weight
            return self.final(fused), center_map, boundary_map

        return self.final(fused)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'pool_size': self.pool_size,
            'filters': self.filters,
            'attention_kernel': self.attention_kernel,
            'use_structure_heads': self.use_structure_heads,
            'n_centers': self.n_centers,
            'center_sharpness': self.center_sharpness,
            'boundary_suppress': self.boundary_suppress,
            'learnable_temperature': self.learnable_temperature
        })
        return cfg

In [28]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers

def split_branch_block(x, filters, dilation_rate=2, l2=1e-4, l1=1e-5):
    c = x.shape[-1]
    assert c % 4 == 0

    reg_k = regularizers.L2(l2)
    reg_b = regularizers.L1(l1)

    # -------------------------------------------------
    # Channel Split + gate
    # -------------------------------------------------
    splits = tf.split(x, num_or_size_splits=4, axis=-1)

    gate_logits = layers.GlobalAveragePooling2D()(x)
    gate_logits = layers.Dense(
        4,
        use_bias=True,
        kernel_regularizer=reg_k,
        bias_regularizer=reg_b
    )(gate_logits)
    gates = layers.Activation("softmax")(gate_logits)

    gated_splits = [
        splits[i] * gates[:, i][:, None, None, None]
        for i in range(4)
    ]

    fpb = filters // 4  # filters per branch, target channel count for every branch

    # -------------------------------------------------
    # Multi-branch convolutions
    # -------------------------------------------------
    b1 = layers.Conv2D(
        fpb, 3, padding="same",
        kernel_regularizer=reg_k,
        bias_regularizer=reg_b
    )(gated_splits[0])

    b2 = layers.Conv2D(
        fpb, 3, padding="same",
        dilation_rate=dilation_rate,
        kernel_regularizer=reg_k,
        bias_regularizer=reg_b
    )(gated_splits[1])

    # depthwise preserves input channel count (c//4), so project with a
    # pointwise conv to fpb after each depthwise so shapes match for concat
    b3 = layers.DepthwiseConv2D(
        3, padding="same",
        depthwise_regularizer=reg_k,
        bias_regularizer=reg_b
    )(gated_splits[2])
    b3 = layers.Conv2D(
        fpb, 1, padding="same",
        kernel_regularizer=reg_k,
        bias_regularizer=reg_b
    )(b3)

    b4 = layers.DepthwiseConv2D(
        3, padding="same",
        dilation_rate=dilation_rate,
        depthwise_regularizer=reg_k,
        bias_regularizer=reg_b
    )(gated_splits[3])
    b4 = layers.Conv2D(
        fpb, 1, padding="same",
        kernel_regularizer=reg_k,
        bias_regularizer=reg_b
    )(b4)

    add_1 = layers.Add()([b1, b4])   # fpb channels
    add_2 = layers.Add()([b2, b3])   # fpb channels
    out = layers.Concatenate()([add_1, add_2])  # filters // 2 channels
    return out

In [29]:
import tensorflow as tf
from tensorflow.keras import layers

@tf.keras.utils.register_keras_serializable(package="Custom")
class RadialWaveEvolutionaryConv(layers.Layer):

    def __init__(self,
                 num_patches=4,
                 num_feature_maps=32,
                 kernel_size=3,
                 **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.num_feature_maps = num_feature_maps
        self.kernel_size = kernel_size

    def build(self, input_shape):

        _, H, W, C = input_shape

        self.H = H
        self.W = W

        self.patch_h = H // self.num_patches
        self.patch_w = W // self.num_patches

        # Decision projection
        self.decision_filters = self.add_weight(
            shape=(C, self.num_feature_maps),
            initializer=tf.keras.initializers.GlorotNormal(),
            trainable=True,
            name="decision_space_filters"
        )

        # Convolution kernel
        self.gaussian_kernel = self.add_weight(
            shape=(self.kernel_size,
                   self.kernel_size,
                   C,
                   self.num_feature_maps),
            initializer=tf.keras.initializers.HeNormal(),
            trainable=True,
            name="gaussian_mutation_kernel"
        )

        # Learnable sigma (no user input)
        self.sigma = self.add_weight(
            shape=(),
            initializer=tf.keras.initializers.Constant(0.1),
            trainable=True,
            name="wave_sigma"
        )

        super().build(input_shape)

    # --------------------------------------------------
    # Pareto (tanh, batch-safe)
    # --------------------------------------------------
    def soft_pareto_score(self, objectives):
        # [B, N, M]

        f_i = tf.expand_dims(objectives, axis=2)  # [B, N, 1, M]
        f_j = tf.expand_dims(objectives, axis=1)  # [B, 1, N, M]

        domination = 0.5 * (1.0 + tf.nn.tanh(f_j - f_i))

        domination = tf.clip_by_value(domination, 1e-6, 1.0)

        domination_prod = tf.reduce_prod(domination, axis=-1)  # [B, N, N]

        score = tf.reduce_sum(domination_prod, axis=-1)  # [B, N]

        return score

    # --------------------------------------------------
    # Call
    # --------------------------------------------------
    def call(self, inputs):

        B = tf.shape(inputs)[0]
        C = tf.shape(inputs)[-1]

        # Patch extraction
        patches = tf.image.extract_patches(
            images=inputs,
            sizes=[1, self.patch_h, self.patch_w, 1],
            strides=[1, self.patch_h, self.patch_w, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        num_total_patches = self.num_patches * self.num_patches
        patch_area = self.patch_h * self.patch_w

        patches = tf.reshape(
            patches,
            [B, num_total_patches, patch_area, C]
        )

        patches_mean = tf.reduce_mean(patches, axis=2)  # [B, N, C]

        # Objectives
        objectives = tf.matmul(patches_mean, self.decision_filters)  # [B, N, M]

        # Pareto
        pareto_score = self.soft_pareto_score(objectives)

        weights = tf.nn.softmax(-pareto_score, axis=-1)
        weights = tf.expand_dims(weights, -1)

        weighted_objectives = objectives * weights

        # Radial wave
        center = tf.reduce_mean(weighted_objectives, axis=1, keepdims=True)

        radial_distance = tf.norm(weighted_objectives - center, axis=-1)

        sigma = tf.nn.softplus(self.sigma) + 1e-6

        # XLA-compatible approximation of Bessel J0 using cosine
        # cos(x / sqrt(2)) closely matches Bessel J0's behavior for small x
        wave = tf.math.cos(radial_distance / 1.41421356) 
        
        decay = tf.exp(- (radial_distance ** 2) / (2.0 * sigma ** 2))


        wave = wave * decay  # [B, N]

        # Expand to spatial
        wave = tf.reshape(wave, [B, self.num_patches, self.num_patches, 1])

        wave = tf.image.resize(
            wave,
            size=(self.H, self.W),
            method="bilinear" # XLA fully supports this for both forward and backward pass
        )


        # Convolution
        conv_out = tf.nn.conv2d(
            inputs,
            self.gaussian_kernel,
            strides=1,
            padding="SAME"
        )

        return conv_out * wave

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "num_feature_maps": self.num_feature_maps,
            "kernel_size": self.kernel_size
        })
        return config

def rwe_conv_block(x, filters, kernel_size=3, padding="same", activation="tanh"):
    x = RadialWaveEvolutionaryConv(num_patches=4, num_feature_maps=filters, kernel_size=kernel_size)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    return x

def build_normal_unet(input_shape=(256, 256, 3), base_filters=64, num_classes=1):
    inputs = layers.Input(shape=input_shape)

    # ===================== Encoder =====================

    # -------- Level 1 --------
    f1 = rwe_conv_block(inputs, base_filters)
    
    p1 = layers.MaxPooling2D(pool_size=(2, 2))(f1)
    p1 = layers.Dropout(0.05)(p1)
    p1 = layers.GaussianDropout(0.1)(p1)

    # -------- Level 2 --------
    f2 = rwe_conv_block(p1, base_filters * 2)
    
    p2 = layers.MaxPooling2D(pool_size=(2, 2))(f2)
    p2 = layers.Dropout(0.15)(p2)
    p2 = layers.GaussianDropout(0.2)(p2)

    # -------- Level 3 --------
    f3 = rwe_conv_block(p2, base_filters * 4)
    
    p3 = layers.MaxPooling2D(pool_size=(2, 2))(f3)
    p3 = layers.Dropout(0.2)(p3)

    # -------- Level 4 --------
    f4 = rwe_conv_block(p3, base_filters * 8)
    
    p4 = layers.MaxPooling2D(pool_size=(2, 2))(f4)
    p4 = layers.Dropout(0.25)(p4)
    p4 = layers.GaussianDropout(0.2)(p4)

    # -------- Bottleneck --------
    f5 = rwe_conv_block(p4, base_filters * 12)
    f5 = layers.SpatialDropout2D(0.2)(f5)
    f5 = layers.GaussianDropout(0.1)(f5)

    # ===================== Decoder =====================

    # -------- Up 4 --------
    # 1. Upsample
    d4 = layers.UpSampling2D(size=(2, 2))(f5)
    
    # 2. Dropouts exactly as specified
    d4 = layers.Dropout(0.05)(d4)
    d4 = layers.GaussianDropout(0.18)(d4)
    
    # 3. Conv block reduces channels to match f4 (base_filters * 8)
    d4 = rwe_conv_block(d4, base_filters * 8)
    
    # 4. Residual Add
    d4 = layers.Add()([f4, d4])

    # -------- Up 3 --------
    d3 = layers.UpSampling2D(size=(2, 2))(d4)
    
    d3 = layers.GaussianDropout(0.18)(d3)
    d3 = layers.Dropout(0.1)(d3)
    
    d3 = rwe_conv_block(d3, base_filters * 4)
    d3 = layers.Add()([f3, d3])

    # -------- Up 2 --------
    d2 = layers.UpSampling2D(size=(2, 2))(d3)
    
    d2 = rwe_conv_block(d2, base_filters * 2)
    d2 = layers.Dropout(0.22)(d2)
    
    d2 = layers.Add()([f2, d2])

    # -------- Up 1 --------
    d1 = layers.UpSampling2D(size=(2, 2))(d2)
    
    d1 = rwe_conv_block(d1, base_filters)
    d1 = layers.Dropout(0.24)(d1)
    
    d1 = layers.Add()([f1, d1])

    # ===================== Output =====================
    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", name="segmentation_output")(d1)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Glas_UNet_RWEConv")

    return model

In [30]:
# Create the Normal UNet model
model = build_normal_unet(input_shape=(256, 256, 3), base_filters=32, num_classes=1)

# Compile using your new Dice+IoU single loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=dice_iou_loss, 
    metrics=['accuracy']
)

# See the summary of your new network
model.summary()

Model: "Glas_UNet_RWEConv"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ radial_wave_evolut… │ (None, 256, 256,  │        961 │ input_layer_1[0]… │
│ (RadialWaveEvoluti… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ radial_wave_evol… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 128, 128,  │          0 │ activation_9[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 128, 128,  │          0 │ max_pooling2d_4[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_dropout_6  │ (None, 128, 128,  │          0 │ dropout_8[0][0]   │
│ (GaussianDropout)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ radial_wave_evolut… │ (None, 128, 128,  │     20,481 │ gaussian_dropout… │
│ (RadialWaveEvoluti… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ radial_wave_evol… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 64, 64,    │          0 │ activation_10[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 64, 64,    │          0 │ max_pooling2d_5[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_dropout_7  │ (None, 64, 64,    │          0 │ dropout_9[0][0]   │
│ (GaussianDropout)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ radial_wave_evolut… │ (None, 64, 64,    │     81,921 │ gaussian_dropout… │
│ (RadialWaveEvoluti… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ radial_wave_evol… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_11       │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 32, 32,    │          0 │ activation_11[0]

 Total params: 2,832,618 (10.81 MB)

 Trainable params: 2,829,930 (10.80 MB)

 Non-trainable params: 2,688 (10.50 KB)

In [31]:
# unet_canvas_full.py
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras import backend as K

# -----------------------------
# Optional imports / fallbacks
# -----------------------------
try:
    from tensorflow.keras.optimizers import AdamW
except Exception:
    AdamW = tf.keras.optimizers.Adam  # fallback to Adam if AdamW not present

# -----------------------------
# Metrics & simple losses
# -----------------------------
@register_keras_serializable(package="Custom")
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    denominator = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])
    dice = (2. * intersection + smooth) / (denominator + smooth)
    return tf.reduce_mean(dice)

@register_keras_serializable(package="Custom")
def dice_loss(y_true, y_pred, smooth=1e-6):
    return 1.0 - dice_coef(y_true, y_pred, smooth)

@register_keras_serializable(package="Custom")
def iou_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=[1,2,3])
    iou = (intersection + smooth) / (union + smooth)
    return tf.reduce_mean(iou)

@register_keras_serializable(package="Custom")
def iou_loss(y_true, y_pred, smooth=1e-6):
    return 1.0 - iou_coef(y_true, y_pred, smooth)

@register_keras_serializable(package="Custom")
def aggregated_jaccard_index(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=[1, 2, 3])
    jaccard = (intersection + smooth) / (union + smooth)

    weights = tf.reduce_sum(y_true, axis=[1, 2, 3]) + smooth
    weighted_mean = tf.reduce_sum(jaccard * weights) / tf.reduce_sum(weights)
    return weighted_mean

@register_keras_serializable(package="Custom")
def f1_score(y_true, y_pred, threshold=0.5, smooth=1e-7):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    true_pos = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    predicted_pos = tf.reduce_sum(y_pred, axis=[1,2,3])
    possible_pos = tf.reduce_sum(y_true, axis=[1,2,3])

    precision = (true_pos + smooth) / (predicted_pos + smooth)
    recall = (true_pos + smooth) / (possible_pos + smooth)
    f1 = 2 * (precision * recall) / (precision + recall + smooth)
    return tf.reduce_mean(f1)

In [32]:
# -----------------------------
# Serializable metrics wrappers
# -----------------------------
@register_keras_serializable(package="Custom")
class IoUMetric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="iou_coef"):
        super().__init__(iou_coef, name=name)

@register_keras_serializable(package="Custom")
class DiceMetric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="dice_coef"):
        super().__init__(dice_coef, name=name)

@register_keras_serializable(package="Custom")
class F1Metric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="f1_score"):
        super().__init__(f1_score, name=name)

@register_keras_serializable(package="Custom")
class AJIMetric(tf.keras.metrics.MeanMetricWrapper):
    def __init__(self, name="aggregated_jaccard_index"):
        super().__init__(aggregated_jaccard_index, name=name)

In [33]:
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable

# -----------------------------
# Simplified LambdaLogger callback
# -----------------------------
@register_keras_serializable(package="Custom")
class LambdaLogger(tf.keras.callbacks.Callback):
    def __init__(self, writer=None):
        super().__init__()
        self.writer = writer

    def on_epoch_end(self, epoch, logs=None):
        if not hasattr(self.model, "lambda1"):
            return

        # Fetch softplus-transformed adaptive weights
        l1 = tf.nn.softplus(self.model.lambda1).numpy()
        l2 = tf.nn.softplus(self.model.lambda2).numpy()
        l3 = tf.nn.softplus(self.model.lambda3).numpy()
        lfg = tf.nn.softplus(self.model.lambda_fg).numpy()
        lbg = tf.nn.softplus(self.model.lambda_bg).numpy()

        # Print clean epoch summary
        msg = (f"Epoch {epoch+1}: λ1={l1:.4f}, λ2={l2:.4f}, λ3={l3:.4f}, "
               f"λ_fg={lfg:.4f}, λ_bg={lbg:.4f}")
        print(msg)

        # Log to TensorBoard if writer is available
        if self.writer is not None:
            with self.writer.as_default():
                tf.summary.scalar("lambda1", l1, step=epoch)
                tf.summary.scalar("lambda2", l2, step=epoch)
                tf.summary.scalar("lambda3", l3, step=epoch)
                tf.summary.scalar("lambda_fg", lfg, step=epoch)
                tf.summary.scalar("lambda_bg", lbg, step=epoch)
                self.writer.flush()

    def get_config(self):
        return {"writer": None}

In [34]:
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable(package="Custom")
class HyperStructuralLoss(tf.keras.losses.Loss):
    """
    Segmentation loss with learnable λ weights.

    Components:
        L_seg = λ2 * Dice + λ3 * IoU

    NOTE: the BCE term, the canvas consistency loss, the patchwise
    discriminator loss, and the center/boundary structural loss
    have all been removed. Only Dice + IoU remain.
    """
    def __init__(self, init_lambdas=None, name="hyper_structural_loss"):
        super().__init__(name=name)
        if init_lambdas is None:
            init_lambdas = {"l2": 3.0, "l3": 4.5}

        # Learnable λ variables
        self.l2 = tf.Variable(init_lambdas["l2"], trainable=True, dtype=tf.float32, name="l2")
        self.l3 = tf.Variable(init_lambdas["l3"], trainable=True, dtype=tf.float32, name="l3")

        # Explicitly expose trainable weights for TensorFlow
        self._trainable_weights = [self.l2, self.l3]

    def call(self, y_true, y_pred):
        """
        Args:
            y_true: ground-truth segmentation mask
            y_pred: predicted segmentation mask
        """
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        # --- Segmentation Loss ---
        L2, L3 = tf.nn.softplus(self.l2), tf.nn.softplus(self.l3)
        intersection = tf.reduce_sum(y_true * y_pred)
        dice = 1.0 - (2.0 * intersection + 1e-6) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1e-6)
        union = tf.reduce_sum(y_true + y_pred) - intersection
        iou = 1.0 - (intersection + 1e-6) / (union + 1e-6)
        seg_loss = L2 * dice + L3 * iou

        return seg_loss

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "init_lambdas": {
                "l2": float(self.l2.numpy()),
                "l3": float(self.l3.numpy())
            }
        })
        return cfg

In [35]:
from tensorflow.keras.optimizers import AdamW
from datetime import datetime

In [36]:
import tensorflow as tf
from glob import glob
from PIL import Image
import numpy as np
import os

# Parameters
img_size = 256
batch_size = 4

# Paths
train_img_paths = sorted(glob("/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/aug_tune/images/*.png"))
train_mask_paths = sorted(glob("/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/aug_tune/masks/*.png"))

val_img_paths = sorted(
    glob("/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/val/images/*.png") +
    glob("/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/val/images/*.tif") +
    glob("/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/val/images/*.tiff")
)
val_mask_paths = sorted(glob("/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/val/masks/*.png"))

print("Train images:", len(train_img_paths))
print("Train masks:", len(train_mask_paths))
print("Val images:", len(val_img_paths))
print("Val masks:", len(val_mask_paths))

# -------------------------------------------------------
# SAFETY CHECK
# -------------------------------------------------------
def stem(p):
    return os.path.splitext(os.path.basename(p))[0]

assert [stem(p) for p in train_img_paths] == [stem(p) for p in train_mask_paths], \
    "❌ Train image-mask mismatch"

assert [stem(p) for p in val_img_paths] == [stem(p) for p in val_mask_paths], \
    "❌ Val image-mask mismatch"

# -----------------------------
# LOADER
# -----------------------------
def load_image_mask(img_path, mask_path):
    img = np.array(
        Image.open(img_path.numpy().decode("utf-8"))
        .convert("RGB")
        .resize((img_size, img_size), Image.BILINEAR)
    ) / 255.0

    mask = np.array(
        Image.open(mask_path.numpy().decode("utf-8"))
        .convert("L")
        .resize((img_size, img_size), Image.NEAREST)
    )

    mask = (mask > 127.5).astype(np.float32)[..., None]
    return img, mask

def tf_load_image_mask(img_path, mask_path):
    img, mask = tf.py_function(
        load_image_mask, [img_path, mask_path], [tf.float32, tf.float32]
    )
    img.set_shape([img_size, img_size, 3])
    mask.set_shape([img_size, img_size, 1])
    return img, mask

# -----------------------------
# DATASETS
# -----------------------------
train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_img_paths, train_mask_paths))
    .map(tf_load_image_mask, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(buffer_size=len(train_img_paths), seed=42)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((val_img_paths, val_mask_paths))
    .map(tf_load_image_mask, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training batches:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_dataset).numpy())

Train images: 350
Train masks: 350
Val images: 2
Val masks: 2
Training batches: 88
Validation batches: 1


In [37]:
class StructTrainer(tf.keras.Model):
    def __init__(self, backbone, loss_fn, optimizer, metrics):
        super().__init__()
        self.backbone = backbone
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.metrics_list = metrics

        # Expose learnable λ weights for monitoring
        for key in ["l2", "l3"]:
            setattr(self, key, getattr(loss_fn, key))

    def compile(self):
        super().compile(optimizer=self.optimizer)

    def train_step(self, data):
        x, y_true = data
        with tf.GradientTape() as tape:
            y_pred = self.backbone(x, training=True)
            loss = self.loss_fn(y_true, y_pred)

        grads = tape.gradient(loss, self.backbone.trainable_variables + self.loss_fn._trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.backbone.trainable_variables + self.loss_fn._trainable_weights))

        logs = {"loss": loss}
        for m in self.metrics_list:
            m.update_state(y_true, y_pred)
            logs[m.name] = m.result()
        return logs

    def test_step(self, data):
        x, y_true = data
        y_pred = self.backbone(x, training=False)
        loss = self.loss_fn(y_true, y_pred)

        logs = {"val_loss": loss}
        for m in self.metrics_list:
            m.update_state(y_true, y_pred)
            logs[f"val_{m.name}"] = m.result()
        return logs

In [38]:
optimizer = AdamW(learning_rate=1e-3, weight_decay=1e-10)

# 3️⃣ Metrics
metrics_list = [
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    #tf.keras.metrics.MeanIoU(num_classes=2, name="iou"),
    IoUMetric(), DiceMetric(), F1Metric(), AJIMetric()
]

# 4️⃣ Loss
loss_fn = HyperStructuralLoss()
ll=LambdaLogger()
# 5️⃣ Trainer
seg_model = StructTrainer(model, loss_fn, optimizer, metrics_list)
seg_model.compile()

# 6️⃣ Callbacks
log_dir = "./logs/" + datetime.now().strftime("%Y%m%d-%H%M%S")
writer = tf.summary.create_file_writer(log_dir)
lambda_logger = LambdaLogger(writer=writer)

early_stp = tf.keras.callbacks.EarlyStopping(
    monitor='val_val_iou_coef', patience=35, mode='max', restore_best_weights=True
)

lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_val_iou_coef', mode='max', factor=0.1, patience=10, min_lr=1e-5, verbose=1
)

tb = tf.keras.callbacks.TensorBoard(log_dir=log_dir)

class SaveBackboneCallback(tf.keras.callbacks.Callback):
    def __init__(self, save_path):
        super().__init__()
        self.save_path = save_path
        self.best_iou = 0.0

    def on_epoch_end(self, epoch, logs=None):
        val_iou = logs.get("val_val_iou_coef", 0)
        if val_iou >= self.best_iou:
            self.best_iou = val_iou
            self.model.backbone.save(self.save_path)
            print(f"\n✅ Saved improved backbone at epoch {epoch+1} with val_iou={val_iou:.4f}")

save_backbone = SaveBackboneCallback("prior_gided_unet.keras")

# 7️⃣ Training
history = seg_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=[early_stp, lr_callback, tb, lambda_logger, save_backbone],batch_size=4,
    verbose=1
)
print("\n🎯 Training complete. Backbone saved and ready for inference.")

Epoch 1/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step - accuracy: 0.7212 - aggregated_jaccard_index: 0.4571 - dice_coef: 0.6009 - f1_score: 0.6194 - iou_coef: 0.4391 - loss: 3.3298
✅ Saved improved backbone at epoch 1 with val_iou=0.5178
88/88 ━━━━━━━━━━━━━━━━━━━━ 87s 511ms/step - accuracy: 0.7614 - aggregated_jaccard_index: 0.5006 - dice_coef: 0.6447 - f1_score: 0.6598 - iou_coef: 0.4832 - loss: 0.0000e+00 - val_val_accuracy: 0.9103 - val_val_aggregated_jaccard_index: 0.5055 - val_val_dice_coef: 0.6811 - val_val_f1_score: 0.7259 - val_val_iou_coef: 0.5178 - val_val_loss: 3.1291 - learning_rate: 0.0010
Epoch 2/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.8197 - aggregated_jaccard_index: 0.5603 - dice_coef: 0.7007 - f1_score: 0.7138 - iou_coef: 0.5439 - loss: 2.7733
✅ Saved improved backbone at epoch 2 with val_iou=0.5220
88/88 ━━━━━━━━━━━━━━━━━━━━ 16s 150ms/step - accuracy: 0.8253 - aggregated_jaccard_index: 0.5664 - dice_coef: 0.7064 - f1_score: 0.7185 - iou_coef: 0.5508 

In [39]:
###### from glob import glob
import os
import tensorflow as tf
import numpy as np
from PIL import Image

# -----------------------------
# CONFIG
# -----------------------------
IMG_DIR = "/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/MoNuSeg_2025_Testing/images"
MSK_DIR = "/kaggle/input/datasets/kartikmaity/tnbcmonusegcpm17kumarcryonuseg-augmentation-datase/MONUSEG_DATA/MONUSEG_DATA/MoNuSeg_2025_Testing/masks"
img_size = 256
batch_size = 6

# -----------------------------
# COLLECT PATHS (SAFE)
# -----------------------------
def valid_name(p):
    return "Zone" not in os.path.basename(p)

image_paths = sorted(
    [p for p in glob(os.path.join(IMG_DIR, "*")) if valid_name(p)]
)

mask_paths = sorted(
    [p for p in glob(os.path.join(MSK_DIR, "*")) if valid_name(p)]
)

print("Images found:", len(image_paths))
print("Masks found :", len(mask_paths))

# -----------------------------
# SAFETY CHECK (CRITICAL)
# -----------------------------
def stem(p):
    s = os.path.splitext(os.path.basename(p))[0]
    for suffix in ["_lesion", "_mask", "_segmentation", "_Segmentation"]:
        s = s.replace(suffix, "")
    return s

assert [stem(p) for p in image_paths] == [stem(p) for p in mask_paths], \
    "❌ Image–mask filename mismatch"

# -----------------------------
# NUMPY LOADER
# -----------------------------
def _load_numpy(img_path, mask_path):
    img_path = img_path.numpy().decode("utf-8")
    mask_path = mask_path.numpy().decode("utf-8")

    img = Image.open(img_path).convert("RGB").resize(
        (img_size, img_size), Image.BILINEAR
    )

    mask = Image.open(mask_path).convert("L").resize(
        (img_size, img_size), Image.NEAREST
    )

    img = np.array(img, dtype=np.float32) / 255.0
    mask = (np.array(mask) > 127.5).astype(np.float32)[..., None]

    return img, mask

# -----------------------------
# TF WRAPPER
# -----------------------------
def load_image_mask(img_path, mask_path):
    img, mask = tf.py_function(
        _load_numpy, [img_path, mask_path], [tf.float32, tf.float32]
    )
    img.set_shape([img_size, img_size, 3])
    mask.set_shape([img_size, img_size, 1])
    return img, mask

# -----------------------------
# DATASET
# -----------------------------
test_dataset = (
    tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    .map(load_image_mask, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

print("Validation batches:", tf.data.experimental.cardinality(test_dataset).numpy())

Images found: 14
Masks found : 14
Validation batches: 3


In [40]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from skimage.color import label2rgb
from tensorflow.keras.optimizers import AdamW
from glob import glob
import os

# ---------------------------------------------------------
# 1️⃣ Import custom layers and metrics (must match training)
# ---------------------------------------------------------
custom_objects = {
    "IoUMetric": IoUMetric,
    "DiceMetric": DiceMetric,
    "F1Metric": F1Metric,
    "AJIMetric": AJIMetric,
    "dice_iou_loss": dice_iou_loss # Make sure to include our custom loss here!
}

# ---------------------------------------------------------
# 2️⃣ Locate + load model
# ---------------------------------------------------------
# Set this to the exact file you want to load, or leave as None to auto-detect.
# Training code saves the backbone as "prior_gided_unet.keras";
# a plain UNet is typically saved as "normal_unet.keras".
MODEL_FILE = None   # e.g. "/kaggle/working/prior_gided_unet.keras"

def find_model():
    if MODEL_FILE is not None:
        return MODEL_FILE

    search_dirs = [d for d in ["/kaggle/working", "/kaggle/input", "."] if os.path.exists(d)]

    # Collect every .keras file in the search directories
    found = []
    for d in search_dirs:
        found += glob(os.path.join(d, "**", "*.keras"), recursive=True)

    print("📂 .keras files found:", found)

    # Preferred names, in order
    for name in ["normal_unet.keras", "prior_gided_unet.keras"]:
        for p in found:
            if os.path.basename(p) == name:
                return p

    # Otherwise, if there is exactly one .keras file, use it
    if len(found) == 1:
        return found[0]

    raise FileNotFoundError(
        "No usable .keras model found. Files in /kaggle/working: "
        f"{os.listdir('/kaggle/working') if os.path.exists('/kaggle/working') else 'N/A'}. "
        "Re-run training, or add the notebook output containing the model as an input."
    )

model_path = find_model()
print("✅ Loading model from:", model_path)

model = tf.keras.models.load_model(
    model_path,
    custom_objects=custom_objects,
    compile=False,
    safe_mode=False
)

# Optional: compile with metrics for evaluation
model.compile(
    optimizer=AdamW(1e-4),
    loss=dice_iou_loss,
    metrics=[IoUMetric(), DiceMetric(), F1Metric(), AJIMetric()]
)

# ---------------------------------------------------------
# 3️⃣ Inference + Save Visualizations
# ---------------------------------------------------------
save_dir = "/kaggle/working/results" if os.path.exists("/kaggle/working/") else "./results"
os.makedirs(save_dir, exist_ok=True)

sample_idx = 0  # global counter for filenames

for batch_idx, (images, masks) in enumerate(test_dataset):
    # Model forward pass - standard UNet returns only y_pred
    y_pred = model(images, training=False)

    preds_bin = tf.cast(y_pred > 0.5, tf.float32)
    batch_size = images.shape[0]

    # Convert masks
    if len(masks.shape) == 4 and masks.shape[-1] > 1:
        masks_label = tf.argmax(masks, axis=-1)
    else:
        masks_label = tf.squeeze(masks, axis=-1)

    if preds_bin.shape[-1] > 1:
        preds_label = tf.argmax(preds_bin, axis=-1)
    else:
        preds_label = tf.squeeze(preds_bin, axis=-1)

    # -------------------------
    # Save all samples in batch
    # -------------------------
    for i in range(batch_size):
        sample_idx += 1  # keep global sample numbering

        # 3 subplots since we removed center/boundary maps
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        img_np = images[i].numpy()
        pred_np = preds_label[i].numpy()
        mask_np = masks_label[i].numpy()

        # 1️⃣ Original image
        axes[0].imshow(img_np)
        axes[0].set_title("Original Image")
        axes[0].axis("off")

        # 2️⃣ Predicted segmentation
        axes[1].imshow(label2rgb(pred_np, bg_label=0))
        axes[1].set_title("Predicted Segmentation")
        axes[1].axis("off")

        # 3️⃣ Ground truth
        axes[2].imshow(label2rgb(mask_np, bg_label=0))
        axes[2].set_title("Ground Truth")
        axes[2].axis("off")

        plt.tight_layout()

        save_path = os.path.join(save_dir, f"sample_{sample_idx:04d}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        print(f"✅ Saved {save_path}")

print(f"\n🎯 All {sample_idx} test images processed and saved to '{save_dir}'")

📂 .keras files found: ['/kaggle/working/prior_gided_unet.keras', './prior_gided_unet.keras']
✅ Loading model from: /kaggle/working/prior_gided_unet.keras
✅ Saved /kaggle/working/results/sample_0001.png
✅ Saved /kaggle/working/results/sample_0002.png
✅ Saved /kaggle/working/results/sample_0003.png
✅ Saved /kaggle/working/results/sample_0004.png
✅ Saved /kaggle/working/results/sample_0005.png
✅ Saved /kaggle/working/results/sample_0006.png
✅ Saved /kaggle/working/results/sample_0007.png
✅ Saved /kaggle/working/results/sample_0008.png
✅ Saved /kaggle/working/results/sample_0009.png
✅ Saved /kaggle/working/results/sample_0010.png
✅ Saved /kaggle/working/results/sample_0011.png
✅ Saved /kaggle/working/results/sample_0012.png
✅ Saved /kaggle/working/results/sample_0013.png
✅ Saved /kaggle/working/results/sample_0014.png

🎯 All 14 test images processed and saved to '/kaggle/working/results'


In [41]:
import numpy as np
from sklearn.metrics import f1_score as sk_f1_score, accuracy_score

# --- Initialize metric lists ---
ious, dices, accuracies, f1s, ajis = [], [], [], [], []

# --- Evaluation Loop ---
for imgs, masks in test_dataset:
    # Forward pass
    preds = model.predict(imgs, verbose=0)

    # Handle multi-output model (segmentation, centers, boundaries)
    if isinstance(preds, (list, tuple)):
        seg_preds = preds[0]       # segmentation output
        # center_maps = preds[1]   # not used for metrics
        # boundary_maps = preds[2] # not used for metrics
    else:
        seg_preds = preds

    # --- Threshold segmentation ---
    preds_bin = (seg_preds > 0.5).astype("float32")

    # --- Convert tensors to numpy safely ---
    masks_np = masks.numpy() if isinstance(masks, tf.Tensor) else masks
    preds_np = preds_bin.astype(np.float32)

    # Ensure shapes match
    if masks_np.shape != preds_np.shape:
        preds_np = tf.image.resize(preds_np, masks_np.shape[1:3]).numpy()

    # --- Flatten for sklearn metrics ---
    masks_flat = masks_np.reshape(-1)
    preds_flat = preds_np.reshape(-1)

    # --- Ensure binary (for safety) ---
    masks_flat = np.round(masks_flat).astype(int)
    preds_flat = np.round(preds_flat).astype(int)

    # --- Tensor-based Metrics ---
    iou_val = iou_coef(masks_np, preds_np).numpy()
    dice_val = dice_coef(masks_np, preds_np).numpy()
    aji_val = aggregated_jaccard_index(masks_np, preds_np).numpy()

    ious.append(iou_val)
    dices.append(dice_val)
    ajis.append(aji_val)

    # --- Classical sklearn metrics ---
    acc = accuracy_score(masks_flat, preds_flat)
    f1 = sk_f1_score(masks_flat, preds_flat, zero_division=1)

    accuracies.append(acc)
    f1s.append(f1)

# --- Compute mean metrics ---
mean_iou = np.mean(ious)
mean_dice = np.mean(dices)
mean_aji = np.mean(ajis)
mean_accuracy = np.mean(accuracies)
mean_f1 = np.mean(f1s)

# --- Print results ---
print("\n===== Segmentation Evaluation =====")
print(f"Mean IoU:              {mean_iou:.4f}")
print(f"Mean Dice Coefficient: {mean_dice:.4f}")
print(f"Mean AJI:              {mean_aji:.4f}")
print(f"Mean Accuracy:         {mean_accuracy:.4f}")
print(f"Mean F1 Score:         {mean_f1:.4f}")


===== Segmentation Evaluation =====
Mean IoU:              0.6409
Mean Dice Coefficient: 0.7802
Mean AJI:              0.6422
Mean Accuracy:         0.8969
Mean F1 Score:         0.7803


In [42]:
import shutil
from IPython.display import FileLink

# Zip the results folder
shutil.make_archive('/kaggle/working/results_images', 'zip', '/kaggle/working/results')

# Creates a clickable download link
FileLink('results_images.zip')

/kaggle/working/results_images.zip